**Ch121a | Module 3: Periodic DFT**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/Module3_Periodic-DFT/notebooks/03_codes_workflow_visualization.ipynb)

# Notebook 3: Codes, Workflow & Visualization

---

## Learning Objectives

- VASP input files: INCAR, POSCAR, POTCAR, KPOINTS
- A typical VASP calculation workflow (relax → SCF → properties)
- Most important INCAR tags and their defaults
- Quantum ESPRESSO as an open-source alternative and its pw.x input structure
- VESTA for crystal structure visualization
- vaspkit (recommended for this class, will save time) for efficient pre/post-processing of VASP calculations
- Materials Project database via mp-api,  pymatgen, direct download

## 1. VASP Input Files

A VASP calculation requires exactly four input files:

### INCAR — Calculation Parameters

The **INCAR** file controls everything: what type of calculation, convergence criteria, electronic settings, output, etc. It is a plain text file of `TAG = VALUE` pairs.

```
SYSTEM = My calculation    ! Human-readable label (ignored by VASP)
ENCUT  = 500               ! Plane-wave cutoff (eV)
EDIFF  = 1E-6              ! Electronic convergence (eV)
NSW    = 100               ! Number of ionic steps (0 = single point)
IBRION = 2                 ! Ionic relaxation algorithm (2=RMM-DIIS)
ISIF   = 3                 ! Relax atoms AND cell shape/volume
ISMEAR = 0                 ! Gaussian smearing
SIGMA  = 0.05              ! Smearing width (eV)
```

### POSCAR — Crystal Structure

Defines the unit cell (lattice vectors) and atomic positions in Cartesian or direct (fractional) coordinates.

```
Si bulk — diamond cubic
1.0
  5.4310  0.0000  0.0000
  0.0000  5.4310  0.0000
  0.0000  0.0000  5.4310
Si
8
Direct
  0.000  0.000  0.000
  ...
```

### POTCAR — Pseudopotentials

Concatenation of PAW pseudopotential files for each element (__in the same order as POSCAR__). Generated by:

```bash
cat $VASP_PP_PATH/PAW_PBE/Si/POTCAR > POTCAR          # single element
cat $VASP_PP_PATH/PAW_PBE/Ti/POTCAR \
    $VASP_PP_PATH/PAW_PBE/O/POTCAR  > POTCAR          # TiO2: Ti then O
```

### KPOINTS — k-point Mesh

```
K-Points
0
Monkhorst-Pack
  8  8  8
  0  0  0
```

## 2. Typical VASP Calculation Workflow

A complete property calculation typically involves multiple steps:

```
Step 1: Ionic relaxation
  INCAR: NSW=100, IBRION=2, ISIF=3, EDIFF=1E-5, EDIFFG=-0.02
  → Obtain CONTCAR (relaxed structure)

Step 2: Self-consistent field (SCF)
  Copy CONTCAR → POSCAR
  INCAR: NSW=0, IBRION=-1, EDIFF=1E-6, LWAVE=.TRUE., LCHARG=.TRUE.
  → Obtain WAVECAR, CHGCAR

Step 3a: DOS calculation
  INCAR: ISTART=1, ICHARG=11 (non-SCF), ISMEAR=-5, NEDOS=2000, LORBIT=11
  → Obtain DOSCAR

Step 3b: Band structure
  KPOINTS: Line-mode k-path through high-symmetry points
  INCAR: ISTART=1, ICHARG=11, LORBIT=11
  → Obtain EIGENVAL (parse with vaspkit or pymatgen)

Step 3c: Ionic relaxation of adsorbate
  Modified POSCAR with adsorbate; selective dynamics to fix substrate
```

### INCAR Tags Reference

| Tag | Default | Meaning |
|-----|---------|---------|
| ENCUT | ENMAX from POTCAR | Plane-wave cutoff (eV) |
| EDIFF | 1E-4 | Electronic convergence (eV) |
| EDIFFG | 10×EDIFF | Force/energy ionic convergence |
| NSW | 0 | Number of ionic steps |
| IBRION | -1 (if NSW=0) | Ionic algorithm: -1=off, 2=RMM-DIIS, 1=RMM |
| ISIF | 2 | Stress/cell: 2=relax atoms only, 3=relax all |
| ISMEAR | 1 | Smearing: 0=Gauss, 1=MP, -5=tetra |
| SIGMA | 0.2 | Smearing width (eV) |
| ISPIN | 1 | 1=non-spin, 2=spin-polarized |
| MAGMOM | 1 per atom | Initial magnetic moments |
| LWAVE | .TRUE. | Write WAVECAR |
| LCHARG | .TRUE. | Write CHGCAR |
| LORBIT | 0 | PDOS: 11=lm-resolved |
| NEDOS | 301 | Number of DOS points |
| ALGO | Normal | Electronic minimizer: Normal, Fast, All |
| PREC | Normal | Precision: Normal, Accurate, High |

## 3. Quantum ESPRESSO — Open-source Alternative

**Quantum ESPRESSO (QE)** is a freely available, community-developed suite for periodic DFT. The main code is `pw.x` (plane-wave SCF).

### pw.x input structure

```fortran
&CONTROL
  calculation = 'scf'        ! scf, relax, vc-relax, nscf, bands
  outdir      = './out/'
  pseudo_dir  = './pseudos/'
  prefix      = 'si'
/
&SYSTEM
  ibrav    = 2               ! FCC lattice (Si)
  celldm(1)= 10.263          ! Lattice constant in Bohr
  nat      = 2               ! Number of atoms
  ntyp     = 1               ! Number of atom types
  ecutwfc  = 40.0            ! PW cutoff (Ry) ≈ ENCUT/13.6 eV/Ry
  ecutrho  = 320.0           ! Charge density cutoff (4× for NCpp)
/
&ELECTRONS
  conv_thr = 1.0d-8          ! SCF convergence (Ry)
/
ATOMIC_SPECIES
  Si  28.0855  Si.pbe-n-kjpaw_psl.1.0.0.UPF
ATOMIC_POSITIONS crystal
  Si  0.00  0.00  0.00
  Si  0.25  0.25  0.25
K_POINTS automatic
  8 8 8  0 0 0
```

**Key difference from VASP**: QE uses Rydbergs (Ry) as the energy unit and separates wavefunctions (ecutwfc) from charge density (ecutrho) cutoffs.
**PSLibrary**: To download PP files, https://pseudopotentials.quantum-espresso.org/legacy_tables


## 4. VESTA — Crystal Structure Visualization

[VESTA](https://jp-minerals.org/vesta/en/) is the standard tool for visualizing and editing crystal structures.

| Task | How in VESTA |
|------|-------------|
| Open structure | File → Open → POSCAR, CIF, XSF, etc. |
| View unit cell | Edit → Lattice Parameters |
| Make supercell | Edit → Bonds → adjust search range |
| Export CIF | File → Export Data → CIF |
| Isosurface (charge) | Edit → Volumetric Data → CHGCAR |
| Slice (density) | Edit → Section |

## 5. vaspkit — Pre/Post-processing

[vaspkit](https://vaspkit.com) provides numbered tasks for common VASP workflows:

| Task ID | Function |
|---------|---------|
| 101 | Generate KPOINTS (MP mesh) |
| 102 | Generate KPOINTS (k-path for bands) |
| 211 | Density of States (from DOSCAR) |
| 212 | Projected DOS (PDOS) |
| 303 | Band structure plotting |
| 311 | Band gap analysis |
| 401 | Total/partial charge density |
| 501 | Optical properties |

Run interactively: `vaspkit` → enter task number, or non-interactively: `vaspkit -task 211`.

## 6. Materials Project API and pymatgen

[pymatgen](https://pymatgen.org) + [mp-api](https://api.materialsproject.org) gives programmatic access to the Materials Project database (> 150,000 calculated structures).

```python
from mp_api.client import MPRester

with MPRester("YOUR_API_KEY") as mpr:
    # Search for TiO2 structures
    docs = mpr.summary.search(
        material_ids=["mp-2657"],  # rutile TiO2
        fields=["material_id", "formula_pretty", "band_gap", "structure"]
    )
    for d in docs:
        print(d.material_id, d.formula_pretty, f"Eg = {d.band_gap:.2f} eV")
        structure = d.structure   # pymatgen Structure object
```

Get a free API key at https://materialsproject.org/api.
You can also download files directly as cif from other repositories:
1. https://www.ccdc.cam.ac.uk/structures/
2. https://www.crystallography.net/cod/
Workaround may be difficult sometimes. 

In [1]:
try:
    from pymatgen.core import Structure, Lattice
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

    # Build Si conventional cell
    a = 5.431   # Angstrom
    latt = Lattice.cubic(a)
    species = ["Si"] * 8
    frac_coords = [
        [0.000, 0.000, 0.000],
        [0.250, 0.250, 0.250],
        [0.500, 0.500, 0.000],
        [0.750, 0.750, 0.250],
        [0.500, 0.000, 0.500],
        [0.750, 0.250, 0.750],
        [0.000, 0.500, 0.500],
        [0.250, 0.750, 0.750],
    ]
    si = Structure(latt, species, frac_coords)

    print(f"Formula      : {si.formula}")
    print(f"Num atoms    : {si.num_sites}")
    print(f"a = b = c    : {si.lattice.a:.4f} Å")
    print(f"α = β = γ    : {si.lattice.alpha:.2f}°")
    print(f"Volume       : {si.volume:.4f} Å³")
    sga = SpacegroupAnalyzer(si)
    print(f"Space group  : {sga.get_space_group_symbol()} ({sga.get_space_group_number()})")
except ImportError:
    print("pymatgen not installed. Install with: pip install pymatgen")
    print("Si conventional cell: a = 5.431 Å, Fd-3m (space group 227), 8 atoms")

Formula      : Si8
Num atoms    : 8
a = b = c    : 5.4310 Å
α = β = γ    : 90.00°
Volume       : 160.1915 Å³
Space group  : Fd-3m (227)


## 7. Further Reading

- [VASP Wiki: INCAR](https://www.vasp.at/wiki/index.php/INCAR)
- [Quantum ESPRESSO documentation](https://www.quantum-espresso.org/Doc/INPUT_PW.html)
- [VESTA manual](https://jp-minerals.org/vesta/archives/VESTA_Manual.pdf)
- [vaspkit manual](https://vaspkit.com/tutorials.html)
- [pymatgen documentation](https://pymatgen.org/docs/)
- [Materials Project API docs](https://api.materialsproject.org)
- Jain et al., *APL Materials* **1**, 011002 (2013) — Materials Project
- Ong et al., *Comput. Mater. Sci.* **68**, 314 (2013) — pymatgen

---
*Ch121a | Caltech | Module 3 — Notebook 3 of 5*